# Keras Neural Network Model Selection

This notebook evaluates a Keras Multi-Layer Perceptron neural network for the binary diabetes target using 5-fold stratified cross-validation. Metrics are computed directly from each validation fold.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [3]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# ==========================================
# 3. Build Keras MLP Neural Network Model
# ==========================================
def build_mlp_model(input_dim):
    model = keras.Sequential([
        layers.InputLayer(shape=(input_dim,)),
        layers.BatchNormalization(axis=-1),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
    )

    return model

In [5]:
# ==========================================
# 4. Run 5-Fold Cross-Validation Across Thresholds
# ==========================================

thresholds = [0.5, 0.4, 0.3, 0.25, 0.2]

fold_metrics = []
fold_predictions = []

for fold_number, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42 + fold_number)

    model = build_mlp_model(input_dim=X_train.shape[1])

    X_fold_train = X_train.iloc[train_idx].to_numpy()
    y_fold_train = y_train.iloc[train_idx].to_numpy()
    X_valid = X_train.iloc[valid_idx].to_numpy()
    y_valid = y_train.iloc[valid_idx].to_numpy()

    negative_count = (y_fold_train == 0).sum()
    positive_count = (y_fold_train == 1).sum()
    class_weight = {
        0: 1.0,
        1: negative_count / positive_count,
    }

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    model.fit(
        X_fold_train,
        y_fold_train,
        epochs=100,
        batch_size=1024,
        validation_split=0.1,
        class_weight=class_weight,
        callbacks=[early_stopping],
        verbose=0,
    )

    y_valid_proba = model.predict(X_valid, batch_size=4096, verbose=0).ravel()

    for threshold in thresholds:
        y_valid_pred = (y_valid_proba >= threshold).astype(int)

        fold_metrics.append({
            "Fold": fold_number,
            "Threshold": threshold,
            "Validation Recall": recall_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation Precision": precision_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F1": f1_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F2": fbeta_score(y_valid, y_valid_pred, beta=2, pos_label=1, zero_division=0),
            "Validation AUPRC": average_precision_score(y_valid, y_valid_proba),
            "Validation AUROC": roc_auc_score(y_valid, y_valid_proba),
            "Predicted Positive Rate": y_valid_pred.mean(),
        })

    fold_predictions.append(pd.DataFrame({
        "Fold": fold_number,
        "y_valid": y_valid,
        "y_valid_proba": y_valid_proba,
    }))

keras_fold_metrics_df = pd.DataFrame(fold_metrics)
keras_cv_predictions_df = pd.concat(fold_predictions, ignore_index=True)

keras_fold_metrics_df.round(6)

,Fold,Threshold,Validation Recall,Validation Precision,Validation F1,Validation F2,Validation AUPRC,Validation AUROC,Predicted Positive Rate
0,1,0.50,0.780053,0.313193,0.446939,0.600905,0.419485,0.810967,0.380389
1,1,0.40,0.863402,0.278189,0.420797,0.607717,0.419485,0.810967,0.474011
2,1,0.30,0.922707,0.248692,0.391788,0.598365,0.419485,0.810967,0.566653
3,1,0.25,0.942654,0.234100,0.375058,0.587198,0.419485,0.810967,0.614987
4,1,0.20,0.958148,0.219987,0.357820,0.573365,0.419485,0.810967,0.665198
5,2,0.50,0.780232,0.314028,0.447818,0.601604,0.428378,0.814318,0.379464
6,2,0.40,0.871416,0.278011,0.421538,0.610709,0.428378,0.814318,0.478716
7,2,0.30,0.921460,0.246182,0.388555,0.595027,0.428378,0.814318,0.571658
8,2,0.25,0.943188,0.232464,0.372997,0.585297,0.428378,0.814318,0.619665
9,2,0.20,0.960997,0.218506,0.356054,0.572156,0.428378,0.814318,0.671699


In [6]:
# ==========================================
# 5. Display Selected Cross-Validation Metrics
# ==========================================

keras_cv_summary_df = (
    keras_fold_metrics_df
    .groupby("Threshold", as_index=False)
    .agg({
        "Validation Recall": "mean",
        "Validation Precision": "mean",
        "Validation F1": "mean",
        "Validation F2": "mean",
        "Validation AUPRC": "mean",
        "Validation AUROC": "mean",
        "Predicted Positive Rate": "mean",
    })
    .rename(columns={
        "Validation Recall": "Validation Recall Mean",
        "Validation Precision": "Validation Precision Mean",
        "Validation F1": "Validation F1 Mean",
        "Validation F2": "Validation F2 Mean",
        "Validation AUPRC": "Validation AUPRC Mean",
        "Validation AUROC": "Validation AUROC Mean",
        "Predicted Positive Rate": "Predicted Positive Rate Mean",
    })
)

default_threshold = 0.5
default_scores = keras_cv_summary_df.loc[
    keras_cv_summary_df["Threshold"] == default_threshold
].copy()
default_scores.insert(0, "Selection Rule", "Default threshold")

best_f2_scores = keras_cv_summary_df.loc[
    [keras_cv_summary_df["Validation F2 Mean"].idxmax()]
].copy()
best_f2_scores.insert(0, "Selection Rule", "Max F2 threshold")

y_valid = keras_cv_predictions_df["y_valid"]
y_valid_proba = keras_cv_predictions_df["y_valid_proba"]
fpr, tpr, roc_thresholds = roc_curve(y_valid, y_valid_proba)
best_tpr_fpr_index = (tpr - fpr).argmax()
best_tpr_fpr_threshold = roc_thresholds[best_tpr_fpr_index]
best_tpr_fpr_pred = (y_valid_proba >= best_tpr_fpr_threshold).astype(int)

best_tpr_fpr_scores = pd.DataFrame([{
    "Selection Rule": "Max TPR-FPR threshold",
    "Threshold": best_tpr_fpr_threshold,
    "Validation Recall Mean": recall_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation Precision Mean": precision_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F1 Mean": f1_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F2 Mean": fbeta_score(y_valid, best_tpr_fpr_pred, beta=2, pos_label=1, zero_division=0),
    "Validation AUPRC Mean": average_precision_score(y_valid, y_valid_proba),
    "Validation AUROC Mean": roc_auc_score(y_valid, y_valid_proba),
    "Predicted Positive Rate Mean": best_tpr_fpr_pred.mean(),
    "TPR - FPR": tpr[best_tpr_fpr_index] - fpr[best_tpr_fpr_index],
}])

keras_selected_metrics_df = pd.concat(
    [default_scores, best_f2_scores, best_tpr_fpr_scores],
    ignore_index=True
)

keras_selected_metrics_df.round(6)


,Selection Rule,Threshold,Validation Recall Mean,Validation Precision Mean,Validation F1 Mean,Validation F2 Mean,Validation AUPRC Mean,Validation AUROC Mean,Predicted Positive Rate Mean,TPR - FPR
0,Default threshold,0.50000,0.783952,0.310511,0.444683,0.600521,0.422833,0.810402,0.385934,NaN
1,Max F2 threshold,0.40000,0.867187,0.275389,0.417957,0.606376,0.422833,0.810402,0.481227,NaN
2,Max TPR-FPR threshold,0.47786,0.806746,0.302429,0.439936,0.604978,0.421425,0.810041,0.407439,0.471291
